# STAGE 1: RADAR SIGNAL PROCESSING

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import glob
import math
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader

# =========================================================
# =========================================================
# STAGE 1: RADAR SIGNAL PROCESSING (FUNCTION & PRE-COMPUTATION)
# =========================================================
# =========================================================

def process_radar_frames(radar_folder, start_frame, num_frames=1, error_log=None):
    """ Reads raw radar frames, applies FFTs, and returns the RD map tensor """
    all_adc = []                         #list not array, the loop below will fill it

    for i in range(start_frame, start_frame + num_frames):
        filename = os.path.join(radar_folder, f"{i:06d}.mat")
        try:
            mat = scipy.io.loadmat(filename)
            adc = mat["adcData"].astype(np.complex128)
            all_adc.append(adc)
        except Exception as e:
            if error_log is not None and filename not in error_log:
                error_log.append(filename)
            if len(all_adc) > 0:
                all_adc.append(all_adc[-1])
            else:
                all_adc.append(np.zeros((128, 255, 4, 2), dtype=np.complex128))

    all_adc = np.array(all_adc)

    # all_adc shape: (num_frames, 128, 255, 4, 2)
    # axis 0: frames
    # axis 1: samples (fast time) — 128
    # axis 2: chirps (slow time) — 255
    # axis 3: receivers          — 4
    # axis 4: transmitters       — 2 (NOT TDM — each TX has its own 255 chirps)

    if len(all_adc.shape) < 5:
        # FIX: output shape updated to (3, 128, 255) to match the corrected pipeline
        return torch.zeros((3, 128, 255), dtype=torch.float32)

    num_frames_actual, M, L, NRX, NTX = all_adc.shape
    # num_frames_actual=30, M=128, L=255, NRX=4, NTX=2

    # step 2: range FFT
    # =================
    win_rng = np.hanning(M)                                 #creates an array with 128 values, window size = number of samples per 1 chirp
    adc_win = all_adc * win_rng[None, :, None, None, None]  #all_adc shape is (1, 128, 255, 4, 2) and win_rng shape is (128), 1D array
                                                            #win_rng[None, :, None, None, None] transforms win_rng to (1, 128, 1, 1, 1)

    # the 128-value window is multiplied with the 128 samples in EVERY chirp, in EVERY frame, for EVERY receiver, for EVERY transmitter
    # this is done to reduce spectral leakage

    R = np.fft.fft(adc_win, axis=1)
    # R shape: (1, 128, 255, 4, 2)

    # remember that adc_win is (num_frames, M, L, NRX, NTX)
    # FFT is applied along axis 1 (M) to convert time samples into range bins (distance)
    # FFT on real signals creates mirrored results — we keep only the first half

    # step 3: virtual antenna reorder
    # ================================
    # TX0 data: R[:, :, :, :, 0]  shape (1, 128, 255, 4)
    # TX1 data: R[:, :, :, :, 1]  shape (1, 128, 255, 4)
    # we simply remap (rx, tx) pairs into 8 virtual antennas
    # no chirp interleaving or demuxing needed


    # step 3: virtual antenna reorder
    virt_N = NRX * NTX      # = 8 virtual antennas
    L_tx   = L              # FIX: L_tx = 255 (not 127) — each TX has ALL 255 chirps

    # R_re shape: (frames, range_bins, chirps, virtual_antennas)
    R_re = np.zeros((num_frames_actual, M, L_tx, virt_N), dtype=np.complex128)

    for tx in range(NTX):
        for rx in range(NRX):
            virt = tx * NRX + rx                # virt = 0..7
            R_re[:, :, :, virt] = R[:, :, :, rx, tx]

    # R_re shape: (1, 128, 256, 8)

    # step 4: doppler FFT
    # ===================
    # FIX: window size is now 255 (full chirp count per TX) instead of 127
    win_dop = np.hanning(L_tx)
    R_re *= win_dop[None, None, :, None]
    RD_cube = np.fft.fftshift(np.fft.fft(R_re, n=256, axis=2), axes=2)

    # different chirps capture how targets move over time
    # FFT over the chirps gives us the velocity
    # FFT of complex signals does not create symmetric results — both halves contain different information
    # RD_cube shape: (1, 128, 256, 8)

    # step 6: coherent integration
    # ==========================================
    RD_coherent = np.sum(RD_cube, axis=0)         # sum across frames → (128, 256, 8)

    #sum across virtual antennas THEN extract Amp/Re/Im
    RD_summed = np.sum(RD_coherent, axis=2)  # (128, 256)

    # channel separation
    # Amp = np.sum(np.abs(RD_coherent), axis=2)
    # Re  = np.real(np.sum(RD_coherent, axis=2))
    # Im  = np.imag(np.sum(RD_coherent, axis=2))

    Amp = np.abs(RD_summed)
    Re  = np.real(RD_summed)
    Im  = np.imag(RD_summed)

    # step 7: Z-score normalization for Neural Network
    Amp_n = (Amp - Amp.mean()) / (Amp.std() + 1e-12)
    Re_n  = (Re  - Re.mean())  / (Re.std()  + 1e-12)
    Im_n  = (Im  - Im.mean())  / (Im.std()  + 1e-12)

    # stack channels: (128, 256, 3)
    RD_norm = np.stack([Amp_n, Re_n, Im_n], axis=2)

    # convert to PyTorch tensor and permute to (Channels, Height, Width)
    RD_tensor = torch.tensor(RD_norm, dtype=torch.float32).permute(2, 0, 1)
    # FIX: output shape is now (3, 128, 256) instead of (3, 128, 127)

    return RD_tensor

# ---------------------------------------------------------
# SCRIPT: PRE-PROCESS AND SAVE TENSORS
# ---------------------------------------------------------
def get_frame_range(radar_folder):
    """Reads .mat filenames and returns (min_idx, max_idx)"""
    mat_files = sorted(glob.glob(os.path.join(radar_folder, "*.mat")))
    if not mat_files:
        return None, None
    indices = []
    for f in mat_files:
        try:
            indices.append(int(os.path.splitext(os.path.basename(f))[0]))
        except ValueError:
            continue
    return (min(indices), max(indices)) if indices else (None, None)


def create_processed_dataset(radar_folder, output_tensor_folder, start_idx, end_idx, num_frames=1, error_log=None):
    """
    Performs pre-computation and saves RD tensors for all samples.
    Run this once before training.
    """
    os.makedirs(output_tensor_folder, exist_ok=True)
    valid_starts = list(range(start_idx, end_idx - num_frames + 2))

    print(f"  → {len(valid_starts)} samples to process (frames {start_idx}..{end_idx})")
    for start_frame in tqdm(valid_starts, desc="  FFT"):
        save_path = os.path.join(output_tensor_folder, f"tensor_{start_frame:06d}.pt")
        if not os.path.exists(save_path):
            RD_tensor = process_radar_frames(radar_folder, start_frame, num_frames, error_log)
            torch.save(RD_tensor, save_path)



def precompute_all_scenarios(root_dir, num_frames=1):
    """
    Iterates over all scenarios, pre-computes RD tensors, and saves them in a new folder.
    - Input: root_dir = "/content/radar_data/Automotive/"
    - Output folder structure:
        /content/radar_data/Automotive/
            ├── scenario_1/
            │   └── radar_raw_frame/
            │       ├── 000001.mat
            │       ├── ...
            ├── scenario_2/
            │   └── radar_raw_frame/
            │       ├── 000001.mat
            │       ├── ...
            └── processed_tensors/
                ├── scenario_1/
                │   ├── tensor_000003.pt
                │   ├── ...
                └── scenario_2/
                    ├── tensor_000003.pt
                    ├── ...
    - Each tensor_000003.pt contains an RD tensor computed from frames 3,4,5 (if num_frames=3)
    - Run precompute_all_scenarios once before training; it will check if tensors exist before recomputing.
    - To recompute a specific scenario, run create_processed_dataset only for that scenario.
    """
    print("=" * 50)
    print(f"Scanning: {root_dir}")
    print("=" * 50)

    scenario_count = 0
    global_error_log = []

    for entry in sorted(os.listdir(root_dir)):
        scenario_path = os.path.join(root_dir, entry)

        if entry == "processed_tensors" or not os.path.isdir(scenario_path):
            continue

        if not os.path.isdir(scenario_path):
            continue

        radar_folder  = os.path.join(scenario_path, "radar_raw_frame")
        output_tensor_folder = os.path.join(root_dir, "processed_tensors", entry)

        if not os.path.isdir(radar_folder):
            continue   # skip if no radar folder

        start_idx, end_idx = get_frame_range(radar_folder)
        if start_idx is None:
            print(f"[SKIP] {entry} — no .mat files")
            continue

        print(f"\n[{scenario_count+1}] {entry}  (frames {start_idx} → {end_idx})")
        create_processed_dataset(radar_folder, output_tensor_folder, start_idx, end_idx, num_frames, error_log=global_error_log)
        scenario_count += 1

    print(f"\n Done! Processed {scenario_count} scenarios.")

    # ==========================================
    # ERROR REPORTING
    # ==========================================
    if len(global_error_log) > 0:
        print("\n" + "⚠️"*20)
        print(f" WARNING: Found {len(global_error_log)} missing/corrupted .mat files!")
        print("⚠️"*20)
        print("List of missing files:")
        for err in global_error_log:
            print(f" - {err}")
        print("="*40)
        print("  Tip: Delete the corresponding CSV (Labels) files for these files so the model doesn't penalize them during training.")
    else:
        print("\n Excellent! No missing .mat files were found in the dataset.")


ROOT_DIR = "/content/radar_data/Automotive/"

#precompute_all_scenarios(ROOT_DIR, num_frames=1)
#create_processed_dataset(radar_folder, output_dir, start_idx=3, end_idx=30, num_frames=1)

# STAGE 2: FEATURE EXTRACTION

In [14]:
# =========================================================
# =========================================================
# STAGE 2: FEATURE EXTRACTION
# =========================================================
# =========================================================

class ConvUnit(nn.Module):
    """ Basic Convolution Unit: Conv2d then BatchNorm then SiLU Activation """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,                               # the number of output channels is the number of different pattern detectors
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False                                 # convolution is followed by BatchNorm, thus, we set bias to False
        )

        self.bn = nn.BatchNorm2d(out_channels)
    # this layer normalizes the output of the convolution across the batch per channel
    # normalizes each of the channels (each feature map) independently
    # shape out is still (batch, out_channels, height, width)

        self.act = nn.SiLU()                           # SiLU is a smooth curve that decides how much a neuron should fire based on its input

    # sigmoid linear unit, activation function that adds non-linearity
    # SiLU(x) = x × sigmoid(x), sigmoid(x) = 1 / (1 + e^(-x))
    # For: 1. very negative numbers, it gives a small negative output
    #      2. numbers near zero, it gives a small output
    #      3. big positive numbers, it gives a big positive output (almost the same as the input)
    # shape out is still (batch, out_channels, height, width), if input is (batch, 64, 32, 64), then output is (batch, 64, 32, 64)


    def forward(self, x):
        x = self.conv(x)                   # step 1: convolution to detect patterns
        x = self.bn(x)                     # step 2: batch normalization to normalize values
        x = self.act(x)                    # step 3: SiLU activation to activate or suppress
        return x


class ResidualBlock(nn.Module):
    """ Standard Residual Block """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvUnit(channels, channels)       # channels in and channels out are same number, two tensors can be added together if they have the same shape
        self.conv2 = ConvUnit(channels, channels)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))            # the network only needs to learn the change (residual)


class CSPResNet(nn.Module):
    """ Cross Stage Partial Network (Lightweight Backbone) """
    def __init__(self, in_channels=128, hidden_channels=64, out_channels=128):
        super().__init__()

        self.branch1 = nn.Sequential(                          # branch 1: convolution,then 2 convolutions // skip connection, parallel
            ConvUnit(hidden_channels, hidden_channels),        # Conv(64)
            ResidualBlock(hidden_channels)                     # ResNet Loop
        )

        self.branch2 = ConvUnit(hidden_channels, hidden_channels)         # branch 2: convolution

        self.fuse_conv = ConvUnit(hidden_channels * 2, out_channels)      # this convolution is carried out after concatenation

    def forward(self, x):
        x1, x2 = torch.chunk(x, 2, dim=1)                            # split into 2 branches
        y1 = self.branch1(x1)                                        # branch 1
        y2 = self.branch2(x2)                                        # branch 2
        return self.fuse_conv(torch.cat([y1, y2], dim=1))            # concatenate then convolut


class RDFeatureExtractor(nn.Module):
    """
    Feature Extraction Backbone
    Outputs intermediate Extracted Features (Ft)

    FIX: input is now (B, 3, 64, 255) instead of (B, 3, 64, 127)
    stage 1: (B, 3,   64, 255) → (B, 64,  32, 128)
    stage 2: (B, 64,  32, 128) → (B, 128, 16,  64)
    stage 3: (B, 128, 16,  64) → (B, 128,  8,  32)
    csp:     (B, 128,  8,  32) → (B, 128,  8,  32)
    """
    """
    FIX: input is now (B, 3, 64, 255) instead of (B, 3, 64, 127)
                        |
                        |
                        |
                        |
                        v
    """
    """
    Feature Extraction Backbone
    Outputs intermediate Extracted Features (Ft)

    NEW FIX: input is now (B, 3, 128, 255)
    stage 1: (B, 3,   128, 255) → (B, 64,  64, 128)
    stage 2: (B, 64,  64,  128) → (B, 128, 32, 64)
    stage 3: (B, 128, 32,  64)  → (B, 128, 16, 32)
    csp:     (B, 128, 16,  32)  → (B, 128, 16, 32)
    """

    def __init__(self):
        super().__init__()
        self.stage1 = ConvUnit(3,   64,  stride=2)
        self.stage2 = ConvUnit(64,  128, stride=2)
        self.stage3 = ConvUnit(128, 128, stride=2)
        self.csp    = CSPResNet(128, 64, 128)

    # (Batch, Channels, Height, Width)
    # shrink the spatial size while capturing more abstract features


    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.csp(x)
        return x

# STAGE 3: FEATURE ENHANCEMENT

In [ ]:
# =========================================================
# =========================================================
# STAGE 3: FEATURE ENHANCEMENT
# =========================================================
# =========================================================

class PositionEmbeddingSine(nn.Module):
    """
    2D Sine-Cosine Positional Encoding
    """
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=True, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats                # half of channel dimension, e.g. 64 for C=128
        self.temperature   = temperature
        self.normalize     = normalize
        if scale is None:
            scale = 2 * math.pi            #2pi because sin and cos are wave functions, one full wave cycle takes exactly 2π to complete (one sweep across)
        self.scale = scale

    def forward(self, x):
        # x shape: (B, C, H, W) = (B, 128, 8, 32)
                                                                                         # x is the input which is (Batch, Channels, Height, Width), (1, 128, 8, 16)
        B, C, H, W = x.shape                                                             # unpacks (1, 128, 8, 16) into B, C, H, W
                                                                                         # B=1, C=128, H=8, W=16
        # cumulative sum gives position index along each axis
        y_embed = torch.ones((B, H, W), dtype=torch.float32).cumsum(1)       # creates a grid of 1 batch, hight of 8 and width of 16 filled entirely with the number 1
                                                                             # cumulative sum, cumsum(1) = travel downward (along rows)
                                                                             # every cell in the same row gets the same row number
                                                                             # y_embed contains row numbers (1, 2, 3 ... 8)
                                                                             # 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
                                                                             # 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
                                                                             # 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
                                                                             # .
                                                                             # .
                                                                             # 7 7 7 7 7 7 7 7 7 7 7 7 7 7 7 7
                                                                             # 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8





        x_embed = torch.ones((B, H, W), dtype=torch.float32).cumsum(2)       # creates a grid of 1 batch, hight of 8 and width of 16 filled entirely with the number 1
                                                                             # cumsum(2) = travel rightward (along columns)
                                                                             # every cell in the same column gets the same column number
                                                                             # x_embed contains columns numbers (1, 2, 3 ... 16)
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16
                                                                             # 1 2 3 ................ 15 16

        if self.normalize:                                                        # normalize to [0, 2pi]
            eps = 1e-6                                                            # protection against dividing by zero

            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale           # [:, -1:, :] is all batches, the last row only, all columns
                                                                                  # we want to stretch our numbers to 2pi = 6.28, scale=6.28
                                                                                  # new value = (old value / maximum value) * 2π, like converting to a percentage

            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale           # [:, :, -1:] is all batches, all rows, the last column only

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)  # creates a list of 64 decimal numbers starting from 0, num_pos_feats = 64
        exponent = 2 * (dim_t // 2) / self.num_pos_feats   # turns the numbers (0.0, 1.0, 2.0, 3.0 ... 63.0) into 64 different frequencies
                                                           # dim_t // 2 is i
                                                           # the integer division dim_t // 2 results in (0.0, 0.0, 1.0, 1.0 ... ,31.0, 31.0)
                                                           # 2 * (dim_t // 2) is 2i
                                                           # 2 * (dim_t // 2) results in (0.0, 0.0, 2.0, 2.0 ... ,62.0, 62.0), even numbers
                                                           # 2 * (dim_t // 2) / 64 is 2i / num_pos_feats
                                                           # 2 * (dim_t // 2) / 64 reults in (0.0, 0.0, 0.03125, 0.03125, 0.0625, 0.0625, ..., 0.96875, 0.96875)

        dim_t = self.temperature ** exponent     # this is 10000^(2i / num_pos_feats)
                                                 # the frequencies are (10000^0.0, 10000^0.0, 10000^0.03125, 10000^0.03125,.....,10000^0.96875, 10000^0.96875)

        dim_t = dim_t.to(x.device)
        y_embed = y_embed.to(x.device)
        x_embed = x_embed.to(x.device)

        pos_y = y_embed[:, :, :, None] / dim_t                            # position/(10000^(2i / num_pos_feats), pos_y shape:  (1, 8, 16, 64)
        pos_x = x_embed[:, :, :, None] / dim_t                            # position/(10000^(2i / num_pos_feats), pos_x shape:  (1, 8, 16, 64)

        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)  # 0::2 starts at index 0, jumps by 2
                                                                                                         # picks channels 0, 2, 4, 6... (32 channels)
                                                                                                         # 1::2 starts at index 1, jumps by 2
                                                                                                         # picks channels 1, 3, 5, 7... (32 channels)
                                                                                                         # even indices takes sin, odd indices takes cos
                                                                                                    # after stack:[[s0, c1], [s2, c3], [s4, c5] ...], 32 pair, 64 values
                                                                                                    # after flatten: [s0, c1, s2, c3, s4, c5 ...], flat list pf 64 values
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(),
                             pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)

        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class MultiHeadSelfAttention(nn.Module):
    """ Multi-Head Self Attention (MSA) """
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads                 # 8 heads
        self.head_dim  = dim // num_heads          # 128 // 8 = 16 per head
        self.scale     = self.head_dim ** -0.5     # 1 / sqrt(16) = 0.25

        self.W_Q  = nn.Linear(dim, dim)
        self.W_K  = nn.Linear(dim, dim)
        self.W_V  = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, C, H, W = x.shape

        # FIX: N is now H*W = 8*32 = 256 tokens instead of 128
        # step 1 — flatten spatial dims into a sequence of tokens
        x_seq       = x.flatten(2).transpose(1, 2)   # (B, N, C) where N = H*W
        B_s, N, C_s = x_seq.shape

        # step 2 — compute Q, K, V in parallel
        Q = self.W_Q(x_seq).reshape(B_s, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_K(x_seq).reshape(B_s, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_V(x_seq).reshape(B_s, N, self.num_heads, self.head_dim).transpose(1, 2)

        # step 3 — dot product Q·Kᵀ then scale by 1/√head_dim
        attn_scores = (Q @ K.transpose(-2, -1)) * self.scale
        # step 4 — softmax to get attention weights
        attn        = attn_scores.softmax(dim=-1)

        # step 5 — multiply weights by V
        y = (attn @ V).transpose(1, 2).reshape(B_s, N, C_s)

        # step 6 — projection to mix all heads together
        return self.proj(y).transpose(1, 2).reshape(B, C, H, W)


class MLPBlock(nn.Module):
    """ MLP / Feed-Forward network """
    def __init__(self, dim, mlp_ratio=4.0, act_layer=nn.SiLU, drop_rate=0.3):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_dim)                # expand features from 128 to 512, 128 × 4 = 512, more neurons means more capacity to find patterns
        self.act = act_layer()                               # activation function used is sigmoid linear unit

        #  Add Dropout to prevent overfitting in the wide (512) layer
        self.drop = nn.Dropout(drop_rate)

        self.fc2 = nn.Linear(hidden_dim, dim)                # features compressed back to 128 to keep the output size the same

    def forward(self, x):
        B, C, H, W = x.shape                                 # save 2D shape for later

        x_seq = x.flatten(2).transpose(1, 2)                 # (B, C, H, W) = (1, 128, 8, 16) to (B, C, HW) = (1, 128, 128) to (B, HW, C) = (1, 128, 128)
                                                             # flatten(2) merges H and W: 8×16 = 128
                                                             # transpose: (B, C, HW) to (B, HW, C)

        x_seq = self.fc1(x_seq)                              # (1, 128, 128) to (1, 128, 512)
        x_seq = self.act(x_seq)                              # (1, 128, 512) to (1, 128, 512)

        # 🚨 Enable Dropout here
        x_seq = self.drop(x_seq)

        x_seq = self.fc2(x_seq)                              # (1, 128, 512) to (1, 128, 128)


        x_seq = x_seq.transpose(1, 2).reshape(B, C, H, W)    # (B, HW, C) = (1, 128, 128) to (B, C, HW) = (1, 128, 128) to (B, C, H, W) = (1, 128, 8, 16)
                                                             # transpose: (B, N, C) to (B, C, N)
                                                             # reshape: splits N=128 back into H=8 and W=16
                                                             # unflatten: (1, 128, 128) to (1, 128, 128) to (1, 128, 8, 16)

        return x_seq


class FeatureEnhancementBlock(nn.Module):
    """ Enhances Extracted Features into Enhanced Features """
    def __init__(self, channels, height, width, num_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.pos_enc = PositionEmbeddingSine(num_pos_feats=channels//2, normalize=True)
        self.ln1     = nn.LayerNorm(channels)
        self.msa     = MultiHeadSelfAttention(dim=channels, num_heads=num_heads)
        self.ln2     = nn.LayerNorm(channels)
        self.mlp     = MLPBlock(dim=channels, mlp_ratio=mlp_ratio, drop_rate=0.3)

    def forward(self, x):
        x = x + self.pos_enc(x)
        y = self.msa(self.ln1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2))
        x = x + y
        z = self.mlp(self.ln2(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2))
        return x + z


class SPPF(nn.Module):
    """ Spatial Pyramid Pooling - Fast """
    def __init__(self, in_channels, pool_size=5):                      #in_channels=128 (passed from DetectHead class), pool_size=5 (5 in width and 5 in height)
        super().__init__()
        self.conv1 = ConvUnit(in_channels, in_channels, kernel_size=1, stride=1, padding=0)       #Properties to this ConvUnit is:
                                                                                                  #(in_channels=128, out_channels=128, kernel_size=1, stride=1, padding=0)


        self.conv2 = ConvUnit(in_channels * 4, in_channels, kernel_size=1, stride=1, padding=0)   #Properties to this ConvUnit is:
                                                                                                  #(in_channels=4*128, out_channels=128, kernel_size=1, stride=1, padding=0)


        self.m = nn.MaxPool2d(kernel_size=pool_size, stride=1, padding=pool_size // 2)       #the padding here adds
                                                                                             #two rows of zeros above, two rows of zeros below
                                                                                             #two colums of zeros to the right, two colums of zeros to the left
                                                                                             #why? the input with no padding is 8 height * 16 width
                                                                                             #this padding will keep the size of the output as the the size of the input
                                                                                             #even after the three maxpools of the SPPF

    def forward(self, x):         #convolution, 3 Maxpools, concatenation, convolution
        x = self.conv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.conv2(torch.cat((x, y1, y2, y3), 1))


class DetectHead(nn.Module):
    """ Transforms features into final predictions """
    def __init__(self, in_channels):                    #in_channels=128, passed from self.head = DetectHead(128) in FullRadarModel class
        super().__init__()
        self.sppf       = SPPF(in_channels)
        self.final_conv = nn.Conv2d(in_channels, 3, kernel_size=1, stride=1, bias=True)

    def forward(self, x):
        x    = self.final_conv(self.sppf(x))
        conf = x[:, 0:1, :, :]   # Confidence
        pos  = x[:, 1:,  :, :]   # Range + velocity offsets
        return torch.cat([conf, pos], dim=1)


class FullRadarModel(nn.Module):
    """ Links Stage 2 with Stage 3 """
    def __init__(self, drop_rate=0.3):
        super().__init__()
        self.backbone = RDFeatureExtractor()
        # FIX: height=8, width=32 to match the new feature map size (8×32 instead of 8×16)
        self.enhance  = FeatureEnhancementBlock(channels=128, height=16, width=32)

        # 🚨 إضافة Dropout كحاجز أخير قبل الـ Head
        self.dropout  = nn.Dropout(p=drop_rate)

        self.head     = DetectHead(128)

    def forward(self, x):
        ft  = self.backbone(x)
        zt  = self.enhance(ft)

        # 🚨 Enable Dropout before final prediction
        # (only active during training, disabled during evaluation and FPGA deployment)
        zt = self.dropout(zt)

        out = self.head(zt)

        conf_logits = out[:, 0:1, :, :]  # Keep it raw because BCEWithLogitsLoss will handle it

        r_pred = out[:, 1:2, :, :]
        v_pred = out[:, 2:3, :, :]

        return r_pred, v_pred, conf_logits  # r_pred, v_pred, conf

# DATASET (DYNAMIC SLIDING WINDOW) & Loss function

In [ ]:
# =========================================================
# DATASET (LOADING PRE-COMPUTED TENSORS)
# =========================================================
import os
import glob
import math
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

class FullRadarDataset(Dataset):
    """ Reads pre-computed Tensors and computes smoothed Labels dynamically """
    def __init__(self, root_dir, allowed_scenarios=None, num_frames=3, fps=30, is_train=False):
        self.num_frames   = num_frames
        self.dt           = 1.0 / fps
        self.H, self.W    = 16, 32  # Downsampled dimensions for the model head
        self.MAX_RANGE    = 28.5
        self.is_train     = is_train
        self.samples      = []  # Stores tuples of (tensor_folder, label_folder, start_frame)

        tensors_root = os.path.join(root_dir, "processed_tensors")

        # Loop over scenarios to extract frames and map them to their corresponding label folders
        for scenario_name in sorted(os.listdir(tensors_root)):
            if allowed_scenarios is not None and scenario_name not in allowed_scenarios:
                continue

            tensor_folder = os.path.join(tensors_root, scenario_name)
            if not os.path.isdir(tensor_folder):
                continue

            label_folder = os.path.join(root_dir, scenario_name, "text_labels")
            if not os.path.isdir(label_folder):
                print(f"[SKIP] {scenario_name} — no text_labels folder found.")
                continue

            # Find all tensor_*.pt files to extract the start_frame index
            pt_files = sorted(glob.glob(os.path.join(tensor_folder, "tensor_*.pt")))
            for pt in pt_files:
                basename    = os.path.splitext(os.path.basename(pt))[0]  # e.g., 'tensor_000003'
                start_frame = int(basename.split("_")[1])                # Extracts '3'
                self.samples.append((tensor_folder, label_folder, start_frame))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tensor_folder, label_folder, start_frame = self.samples[idx]

        # 1. Load pre-computed Tensor (Single Frame)
        tensor_path = os.path.join(tensor_folder, f"tensor_{start_frame:06d}.pt")
        if os.path.exists(tensor_path):
            RD_tensor = torch.load(tensor_path)
        else:
            # Fallback tensor (matches the 256 Doppler bins from RTL)
            RD_tensor = torch.zeros((3, 128, 256), dtype=torch.float32)

        # Flag to track if velocity needs to be inverted in the labels
        flip_doppler = False

        # ==========================================
        # DATA AUGMENTATION (Applied only during training)
        # ==========================================
        if self.is_train:

            # 1. Random Phase Rotation (50% chance)
            if torch.rand(1).item() > 0.5:
                theta = torch.empty(1).uniform_(0, 2 * math.pi).item()
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)

                # Clone Real and Imaginary channels (Amplitude in channel 0 remains unaffected)
                Re = RD_tensor[1, :, :].clone()
                Im = RD_tensor[2, :, :].clone()

                # Apply rotation matrix
                RD_tensor[1, :, :] = Re * cos_t - Im * sin_t
                RD_tensor[2, :, :] = Re * sin_t + Im * cos_t

            # 2. Doppler Inversion / Horizontal Flip (50% chance)
            if torch.rand(1).item() > 0.5:
                # Flip along the Doppler axis (Width / axis 2)
                RD_tensor = torch.flip(RD_tensor, dims=[2])
                flip_doppler = True  # Activate flag to invert label velocity later

            # 3. Random Cutout / Erasing (50% chance)
            if torch.rand(1).item() > 0.5:
                num_patches = torch.randint(1, 4, (1,)).item() # 1 to 3 patches
                for _ in range(num_patches):
                    # Patch dimensions (5 to 15 pixels)
                    h_size = torch.randint(5, 15, (1,)).item()
                    w_size = torch.randint(5, 15, (1,)).item()

                    # Random coordinates for the patch
                    y0 = torch.randint(0, RD_tensor.shape[1] - h_size, (1,)).item()
                    x0 = torch.randint(0, RD_tensor.shape[2] - w_size, (1,)).item()

                    # Zero out the region across all channels to simulate signal loss
                    RD_tensor[:, y0:y0+h_size, x0:x0+w_size] = 0

            # 4. Random Gaussian Noise (50% chance)
            if torch.rand(1).item() > 0.5:
                noise_std = 0.05
                noise = torch.randn_like(RD_tensor) * noise_std
                RD_tensor = RD_tensor + noise

            # 5. Random Intensity Scaling (50% chance)
            if torch.rand(1).item() > 0.5:
                scale = torch.empty(1).uniform_(0.8, 1.2).item()
                RD_tensor = RD_tensor * scale
        # ==========================================

        # Initialize target output masks
        conf_map = torch.zeros((1, self.H, self.W))
        r_map    = torch.zeros((1, self.H, self.W))
        v_map    = torch.zeros((1, self.H, self.W))
        w_map    = torch.zeros((1, self.H, self.W))
        l_map    = torch.zeros((1, self.H, self.W))

        # 2. Process Labels using Temporal Smoothing
        tracks = {}  # target_id -> list of (px, py, vel, wid, length)

        for i in range(start_frame, start_frame + self.num_frames):
            csv_path = os.path.join(label_folder, f"{i:010d}.csv")
            if not os.path.exists(csv_path):
                continue
            try:
                df = pd.read_csv(csv_path, header=None)
                has_velocity = df.shape[1] >= 7
                for row_idx in range(len(df)):
                    try:
                        target_id = int(df.iloc[row_idx, 0])
                        px        = float(df.iloc[row_idx, 2])
                        py        = float(df.iloc[row_idx, 3])
                        wid       = float(df.iloc[row_idx, 4])
                        length    = float(df.iloc[row_idx, 5])
                        vel       = float(df.iloc[row_idx, 6]) if has_velocity else 0.0

                        #  Appended 'length' correctly instead of duplicating 'vel'
                        tracks.setdefault(target_id, []).append((px, py, vel, wid, length))
                    except:
                        continue
            except Exception:
                pass

        if len(tracks) == 0:
            return RD_tensor, r_map, v_map, conf_map, w_map, l_map

        # Velocity Constants
        MAX_VEL = 16.6
        VEL_OFFSET = 8.3

        # ==========================================
        #  Grid Encoding (Direct & Fast)
        # ==========================================
        for _, seq in tracks.items():
            if len(seq) == 0:
                continue

            r_list = [np.sqrt(px**2 + py**2) for px, py, v, w, l in seq]
            v_list = [v for px, py, v, w, l in seq]
            w_list = [w for px, py, v, w, l in seq]
            l_list = [l for px, py, v, w, l in seq]

            # Average targets over num_frames to smooth out labeling noise
            r_target = float(np.mean(r_list))
            v_mean   = float(np.mean(v_list))
            w_target = float(np.mean(w_list))
            l_target = float(np.mean(l_list))

            #  Apply velocity inversion if Doppler was flipped during augmentation
            if flip_doppler:
                v_mean = -v_mean

            # Clip velocities to protect the model from outlier data
            v_mean = float(np.clip(v_mean, -MAX_VEL, MAX_VEL))

            # Map physical values to grid indices
            range_bin   = max(0, min(self.H - 1, int((r_target / self.MAX_RANGE) * (self.H - 1))))
            doppler_bin = max(0, min(self.W - 1, int(((v_mean + VEL_OFFSET) / MAX_VEL) * (self.W - 1))))

            # Assign values to the mapped bins
            if conf_map[0, range_bin, doppler_bin] == 0:
                conf_map[0, range_bin, doppler_bin] = 1.0
                r_map[0, range_bin, doppler_bin]    = r_target / self.MAX_RANGE
                v_map[0, range_bin, doppler_bin]    = (v_mean + VEL_OFFSET) / MAX_VEL
                w_map[0, range_bin, doppler_bin]    = w_target
                l_map[0, range_bin, doppler_bin]    = l_target

        return RD_tensor, r_map, v_map, conf_map, w_map, l_map

# =========================================================
# LOSS FUNCTION
# =========================================================
class InDTLoss(nn.Module):
    def __init__(self, lambda_pos=0.7, lambda_conf=0.3, pos_weight_val=15.0):
        super().__init__()
        self.lambda_pos = lambda_pos
        self.lambda_conf = lambda_conf
        self.pos_weight_val = pos_weight_val

    def forward(self, r_pred, v_pred, conf_logits, r_true, v_true, conf_true):
        # 1. Binary Cross-Entropy Loss for confidence prediction
        weight_tensor = torch.tensor([self.pos_weight_val], device=conf_logits.device)
        Lc = F.binary_cross_entropy_with_logits(conf_logits, conf_true, pos_weight=weight_tensor)

        obj_mask = conf_true > 0
        if obj_mask.sum() > 0:
            # 2. MSE (L2) Loss for range and velocity predictions (only evaluated at positive target locations)
            loss_r = F.mse_loss(r_pred[obj_mask], r_true[obj_mask])
            loss_v = F.mse_loss(v_pred[obj_mask], v_true[obj_mask])
            Lp = loss_r + loss_v
        else:
            # Zero-loss dummy calculation to prevent NaN and keep the gradient graph connected
            Lp = (r_pred.sum() + v_pred.sum()) * 0.0

        total_loss = (self.lambda_pos * Lp) + (self.lambda_conf * Lc)
        return total_loss, Lp, Lc

# SETUP & TRAINING

In [ ]:
# =========================
# SETUP & TRAINING
# =========================
import torch
import os
import random
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Training on: {device}")

ROOT_DIR = "/content/radar_data/Automotive/"

# Fixed scenarios to avoid data leakage (Train / Val / Test Split)
train_scenarios = [
    '2019_04_30_pbss000', '2019_04_30_pcms001', '2019_04_09_pms2000',
    '2019_05_29_mlms006', '2019_05_29_bcms000', '2019_05_09_mlms003',
    '2019_05_09_bm1s007', '2019_04_30_mlms000', '2019_04_30_pbms002',
    '2019_05_09_pbms004', '2019_04_09_cms1000', '2019_05_09_pcms002',
    '2019_05_29_pcms005', '2019_05_29_cm1s014', '2019_05_29_pbms007'

]
val_scenarios  = ['2019_04_09_pms1000', '2019_04_09_pms3000']
test_scenarios = ['2019_04_09_bms1000', '2019_04_09_css1000']

print(f" Scenarios -> Train: {len(train_scenarios)} | Val: {len(val_scenarios)} | Test: {len(test_scenarios)}")
print(" Initializing Dataset...")

# 1. Initialize Datasets (Ensure num_frames=3 for temporal smoothing as discussed)
train_dataset = FullRadarDataset(ROOT_DIR, allowed_scenarios=train_scenarios, num_frames=3, is_train=True)
val_dataset   = FullRadarDataset(ROOT_DIR, allowed_scenarios=val_scenarios,   num_frames=3, is_train=False)
test_dataset  = FullRadarDataset(ROOT_DIR, allowed_scenarios=test_scenarios,  num_frames=3, is_train=False)

print(f" Total samples -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# 2. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1,   shuffle=False)

# 3. Model, Loss, and Optimizer
model     = FullRadarModel().to(device)
criterion = InDTLoss()

optimizer = torch.optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 5e-5},
    {'params': model.enhance.parameters(),  'lr': 5e-5},
    {'params': model.head.parameters(),     'lr': 5e-5},
], weight_decay=1e-4)

# 4. Training Loop Config
num_epochs = 70
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

print("\n" + "="*50)
print("  Starting Training Phase")
print("="*50)

best_val_loss = float('inf')
patience_counter = 0
patience_limit = 15

for epoch in range(num_epochs):
    # ---------------------------
    # 1. Training Phase
    # ---------------------------
    model.train()
    epoch_loss = 0

    for RD_tensor, r_true, v_true, conf_true, w_true, l_true in train_loader:
        # Move data to GPU/CPU
        RD_tensor = RD_tensor.to(device)
        r_true    = r_true.to(device)
        v_true    = v_true.to(device)
        conf_true = conf_true.to(device)

        # Forward pass
        r_pred, v_pred, conf_logits = model(RD_tensor)

        # Loss calculation
        loss, Lp, Lc = criterion(r_pred, v_pred, conf_logits, r_true, v_true, conf_true)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # ---------------------------
    # 2. Validation Phase
    # ---------------------------
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for RD_tensor, r_true, v_true, conf_true, w_true, l_true in val_loader:
            RD_tensor = RD_tensor.to(device)
            r_true    = r_true.to(device)
            v_true    = v_true.to(device)
            conf_true = conf_true.to(device)

            r_pred, v_pred, conf_logits = model(RD_tensor)
            loss, Lp, Lc = criterion(r_pred, v_pred, conf_logits, r_true, v_true, conf_true)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step()

    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # ---------------------------
    # 3. Early Stopping & Checkpointing
    # ---------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'adas_radar_weights_best.pth')
        print("    Best model saved!")
    else:
        patience_counter += 1
        print(f"    No improvement. Patience: {patience_counter}/{patience_limit}")

    if patience_counter >= patience_limit:
        print("\n Early stopping triggered! Training stopped to prevent overfitting.")
        break

print("\n" + "="*50)
print("  Training completed. Best weights stored in 'adas_radar_weights_best.pth'")
print("="*50)

# =========================================================
# TESTING PHASE (SINGLE FRAME ANALYSIS)
# =========================================================
print("\n" + "="*50)
print("  Start Testing Phase (Single Frame)")
print("="*50)

model.eval()
with torch.no_grad():
    # Load one batch (batch_size=1) from the test loader
    test_tensor, r_true, v_true, conf_true_test, w_true, l_true = next(iter(test_loader))

    test_tensor    = test_tensor.to(device)
    r_true         = r_true.to(device)
    v_true         = v_true.to(device)
    conf_true_test = conf_true_test.to(device)

    # Model inference
    r_pred_test, v_pred_test, conf_logits_test = model(test_tensor)
    conf_probs = torch.sigmoid(conf_logits_test)

    # Radar Limits & Thresholds
    MAX_RANGE  = 28.5
    MAX_VEL    = 16.6
    VEL_OFFSET = 8.3
    CONF_THRESH = 0.50     # Lowered slightly to ensure targets are caught, NMS will handle noise
    RANGE_TOLERANCE = 1.0  # meters
    VEL_TOLERANCE   = 2.0  # m/s

    #  BUG FIX: Local Max Pooling (NMS) to remove duplicate overlapping predictions
    local_max = F.max_pool2d(conf_probs, kernel_size=3, stride=1, padding=1)
    peak_mask = (conf_probs == local_max) & (conf_probs > CONF_THRESH)

    pred_indices = peak_mask.nonzero()
    true_indices = torch.nonzero(conf_true_test > 0.5)

    print(f" Ground Truth Targets: {len(true_indices)}")
    print(f" Predicted Targets   : {len(pred_indices)}\n")
    print("-" * 50)

    matched_true_indices = set()

    # 1. Evaluate Model Predictions
    for idx, p_coord in enumerate(pred_indices):
        p_y, p_x = p_coord[2].item(), p_coord[3].item() # Extracted from Batch & Channel dimensions
        conf_val = conf_probs[0, 0, p_y, p_x].item()

        pred_r = r_pred_test[0, 0, p_y, p_x].item() * MAX_RANGE
        pred_v = (v_pred_test[0, 0, p_y, p_x].item() * MAX_VEL) - VEL_OFFSET

        print(f" PRED {idx + 1} | Conf: {conf_val * 100:.1f}% | Grid: ({p_y:02d}, {p_x:02d})")
        print(f"   Pred Range: {pred_r:>5.2f} m | Pred Vel: {pred_v:>5.2f} m/s")

        # Find the best matching ground truth target
        best_match_idx = -1
        best_match_dist = float('inf')
        best_true_r, best_true_v = 0.0, 0.0

        for t_idx, t_coord in enumerate(true_indices):
            if t_idx in matched_true_indices:
                continue

            t_y, t_x = t_coord[2].item(), t_coord[3].item()
            true_r = r_true[0, 0, t_y, t_x].item() * MAX_RANGE
            true_v = (v_true[0, 0, t_y, t_x].item() * MAX_VEL) - VEL_OFFSET

            r_err = abs(pred_r - true_r)
            v_err = abs(pred_v - true_v)

            if r_err <= RANGE_TOLERANCE and v_err <= VEL_TOLERANCE:
                dist = (r_err**2 + v_err**2)**0.5
                if dist < best_match_dist:
                    best_match_dist = dist
                    best_match_idx = t_idx
                    best_true_r, best_true_v = true_r, true_v

        if best_match_idx != -1:
            matched_true_indices.add(best_match_idx)
            print(f"    True Positive -> Matched Target: Range {best_true_r:.2f} m, Vel {best_true_v:.2f} m/s")
        else:
            print(f"    False Positive -> No un-matched real target within tolerance.")
        print("-" * 50)

    # 2. Print missed ground truth targets (False Negatives)
    for t_idx, t_coord in enumerate(true_indices):
        if t_idx not in matched_true_indices:
            t_y, t_x = t_coord[2].item(), t_coord[3].item()
            true_r = r_true[0, 0, t_y, t_x].item() * MAX_RANGE
            true_v = (v_true[0, 0, t_y, t_x].item() * MAX_VEL) - VEL_OFFSET

            print(f" MISSED TARGET (False Negative) | Grid: ({t_y:02d}, {t_x:02d})")
            print(f"    True Range: {true_r:>5.2f} m | True Vel: {true_v:>5.2f} m/s")
            print("-" * 50)

    if len(pred_indices) == 0 and len(true_indices) == 0:
        print(" True Negative: Frame is empty, and model correctly predicted nothing.")

print("="*50)

 Training on: cuda
 Scenarios -> Train: 15 | Val: 2 | Test: 2
 Initializing Dataset...
 Total samples -> Train: 13470 | Val: 1796 | Test: 1794

  Starting Training Phase
Epoch [01/70] | Train Loss: 0.3818 | Val Loss: 0.2457
    Best model saved!
Epoch [02/70] | Train Loss: 0.2422 | Val Loss: 0.1963
    Best model saved!
Epoch [03/70] | Train Loss: 0.2025 | Val Loss: 0.1677
    Best model saved!
Epoch [04/70] | Train Loss: 0.1747 | Val Loss: 0.1451
    Best model saved!
Epoch [05/70] | Train Loss: 0.1530 | Val Loss: 0.1253
    Best model saved!
Epoch [06/70] | Train Loss: 0.1359 | Val Loss: 0.1099
    Best model saved!
Epoch [07/70] | Train Loss: 0.1212 | Val Loss: 0.0946
    Best model saved!
Epoch [08/70] | Train Loss: 0.1088 | Val Loss: 0.0837
    Best model saved!
Epoch [09/70] | Train Loss: 0.0985 | Val Loss: 0.0782
    Best model saved!
Epoch [10/70] | Train Loss: 0.0896 | Val Loss: 0.0719
    Best model saved!
Epoch [11/70] | Train Loss: 0.0822 | Val Loss: 0.0669
    Best model s

# Test


In [23]:
import torch
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import precision_recall_curve, auc
from tqdm import tqdm
from torch.utils.data import DataLoader

# =========================================================
# 1. SETUP & DATALOADERS
# =========================================================
ROOT_DIR = "/content/radar_data/Automotive/"

train_scenarios = [
    '2019_04_30_pbss000', '2019_04_30_pcms001', '2019_04_09_bms1000',
    '2019_05_29_mlms006', '2019_05_29_bcms000', '2019_05_09_mlms003',
    '2019_05_09_bm1s007', '2019_04_30_mlms000', '2019_04_30_pbms002',
    '2019_05_09_pbms004', '2019_04_09_cms1000', '2019_05_09_pcms002',
    '2019_05_29_pcms005', '2019_05_29_cm1s014', '2019_05_29_pbms007',
    '2019_04_09_pms2000'
]
val_scenarios  = ['2019_04_09_pms1000','2019_04_09_pms3000']
test_scenarios = ['2019_04_09_bms1000','2019_04_09_cms1000']

# Note: num_frames=3 is kept for Temporal Smoothing in labels
train_dataset = FullRadarDataset(ROOT_DIR, allowed_scenarios=train_scenarios, num_frames=3, is_train=True)
val_dataset   = FullRadarDataset(ROOT_DIR, allowed_scenarios=val_scenarios,   num_frames=3, is_train=False)
test_dataset  = FullRadarDataset(ROOT_DIR, allowed_scenarios=test_scenarios,  num_frames=3, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1,  shuffle=False)

# =========================================================
# 2. OLS EVALUATION SETUP & CONSTANTS
# =========================================================
print("\n" + "="*65)
print(" 🚀 STARTING OLS-BASED METRICS EVALUATION ...")
print("="*65)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Model
try:
    model = FullRadarModel().to(device)
    model.load_state_dict(torch.load('adas_radar_weights_best.pth', map_location=device))
    print(" ✅ Successfully loaded weights from 'adas_radar_weights_best.pth'")
except Exception as e:
    print(f" ❌ Error loading model: {e}")

# --- Constants & Thresholds ---
CONF_THRESH = 0.80
MAX_RANGE   = 28.5
MAX_VEL     = 16.6
VEL_OFFSET  = 8.3

# OLS Kappa values from RODNet/RAMP-CNN
KAPPA_DICT = {
    0: 0.5,  # Pedestrian
    1: 0.5,  # Cyclist
    2: 0.9   # Car
}

# Standard Evaluation Thresholds for AP/AR
OLS_THRESHOLDS = np.arange(0.50, 0.95, 0.05)  # [0.50, 0.55, ..., 0.90]

def compute_ols(d, s, kappa):
    """
    Computes Object Location Similarity (OLS)
    d: distance between prediction and ground truth (meters)
    s: range (distance of the object from radar in meters)
    kappa: class-specific tolerance
    """
    epsilon = 1e-9 # To prevent division by zero
    return np.exp((- (d ** 2)) / (2 * ((s * kappa) + epsilon) ** 2))

# --- Trackers for multiple thresholds ---
metrics_per_thresh = {thresh: {'TP': 0, 'FP': 0, 'FN': 0, 'y_scores': [], 'tp_list': []} for thresh in OLS_THRESHOLDS}

# --- System-Level Counters ---
WINDOW_SIZE = 5
sys_metrics = {'sys_TP': 0, 'sys_FP': 0, 'sys_FN': 0, 'sys_TN': 0}
win_frames, win_fps = 0, 0
win_has_gt, win_has_tp = False, False

model.eval()

with torch.no_grad():
    # Note: cls_true is included. Ensure your dataloader yields it!
    for test_tensor, r_true, v_true, conf_true, cls_true, l_true in tqdm(test_loader, desc="Evaluating Frames"):

        test_tensor = test_tensor.to(device)
        r_true      = r_true.to(device)
        v_true      = v_true.to(device)
        conf_true   = conf_true.to(device)
        cls_true    = cls_true.to(device) # We need the class ID for Kappa

        # Model Inference
        r_pred, v_pred, conf_logits = model(test_tensor)
        conf_probs = torch.sigmoid(conf_logits)

        # NMS to find Peaks
        local_max = F.max_pool2d(conf_probs, kernel_size=5, stride=1, padding=2)
        peak_mask = (conf_probs == local_max) & (conf_probs > CONF_THRESH)

        batch_size = test_tensor.size(0)

        for b in range(batch_size):
            c_map_pred = conf_probs[b, 0]
            c_map_true = conf_true[b, 0]

            true_indices = torch.nonzero(c_map_true > 0.5)
            pred_indices = torch.nonzero(peak_mask[b, 0])

            num_true, num_pred = len(true_indices), len(pred_indices)

            # --- OLS Evaluation across all thresholds ---
            for thresh in OLS_THRESHOLDS:
                matched_pred_indices = set()

                for t_coord in true_indices:
                    t_y, t_x = t_coord[0].item(), t_coord[1].item()

                    true_r = r_true[b, 0, t_y, t_x].item() * MAX_RANGE
                    t_class = int(cls_true[b, 0, t_y, t_x].item())
                    kappa = KAPPA_DICT.get(t_class, 0.9) # Default to 0.9 if unknown

                    best_match_idx  = -1
                    best_match_ols  = -1.0
                    best_match_conf = 0.0

                    for p_idx, p_coord in enumerate(pred_indices):
                        if p_idx in matched_pred_indices: continue

                        p_y, p_x = p_coord[0].item(), p_coord[1].item()

                        normalized_r = max(0.0, min(1.0, r_pred[b, 0, p_y, p_x].item()))
                        pred_r = normalized_r * MAX_RANGE
                        pred_conf = c_map_pred[p_y, p_x].item()

                        # Distance calculation (d) in meters
                        d = abs(pred_r - true_r)
                        s = true_r  # Distance from radar

                        # Calculate OLS
                        ols_score = compute_ols(d, s, kappa)

                        # Match if OLS > threshold and it's the highest OLS so far
                        if ols_score >= thresh and ols_score > best_match_ols:
                            best_match_ols  = ols_score
                            best_match_idx  = p_idx
                            best_match_conf = pred_conf

                    if best_match_idx != -1:
                        metrics_per_thresh[thresh]['TP'] += 1
                        matched_pred_indices.add(best_match_idx)
                        metrics_per_thresh[thresh]['y_scores'].append(best_match_conf)
                        metrics_per_thresh[thresh]['tp_list'].append(1)
                        if thresh == 0.50: win_has_tp = True # Flag for system metrics
                    else:
                        metrics_per_thresh[thresh]['FN'] += 1
                        missed_conf = c_map_pred[t_y, t_x].item()
                        metrics_per_thresh[thresh]['y_scores'].append(missed_conf)
                        metrics_per_thresh[thresh]['tp_list'].append(1)

                # Record False Positives for this threshold
                for p_idx, p_coord in enumerate(pred_indices):
                    if p_idx not in matched_pred_indices:
                        metrics_per_thresh[thresh]['FP'] += 1
                        p_y, p_x = p_coord[0].item(), p_coord[1].item()
                        metrics_per_thresh[thresh]['y_scores'].append(c_map_pred[p_y, p_x].item())
                        metrics_per_thresh[thresh]['tp_list'].append(0)
                        if thresh == 0.50: win_fps += 1

            # System-Level Window Check (using baseline threshold 0.50)
            if num_true > 0: win_has_gt = True
            if num_true == 0 and num_pred == 0:
                sys_metrics['sys_TN'] += 1
                win_frames += 1
            else:
                win_frames += 1

            if win_frames == WINDOW_SIZE:
                if win_has_gt:
                    if win_has_tp: sys_metrics['sys_TP'] += 1
                    else: sys_metrics['sys_FN'] += 1
                else:
                    if win_fps > 0: sys_metrics['sys_FP'] += 1
                    else: sys_metrics['sys_TN'] += 1

                win_frames, win_fps = 0, 0
                win_has_gt, win_has_tp = False, False

# Catch remaining window frames
if win_frames > 0:
    if win_has_gt:
        if win_has_tp: sys_metrics['sys_TP'] += 1
        else: sys_metrics['sys_FN'] += 1
    else:
        if win_fps > 0: sys_metrics['sys_FP'] += 1
        else: sys_metrics['sys_TN'] += 1

# =========================================================
# 3. METRICS AGGREGATION & REPORTING
# =========================================================
ap_list = []
ar_list = []
epsilon = 1e-7

for thresh in OLS_THRESHOLDS:
    TP = metrics_per_thresh[thresh]['TP']
    FP = metrics_per_thresh[thresh]['FP']
    FN = metrics_per_thresh[thresh]['FN']

    recall = TP / (TP + FN + epsilon)
    ar_list.append(recall)

    # Calculate AP using Precision-Recall curve
    tp_list = metrics_per_thresh[thresh]['tp_list']
    y_scores = metrics_per_thresh[thresh]['y_scores']

    if len(tp_list) > 0:
        precision_curve, recall_curve, _ = precision_recall_curve(tp_list, y_scores)
        ap = auc(recall_curve, precision_curve)
    else:
        ap = 0.0

    ap_list.append(ap)

# Calculate final Average Precision (mAP) and Average Recall (AR)
final_AP = np.mean(ap_list)
final_AR = np.mean(ar_list)

Sys_Precision = sys_metrics['sys_TP'] / (sys_metrics['sys_TP'] + sys_metrics['sys_FP'] + epsilon)
Sys_Recall    = sys_metrics['sys_TP'] / (sys_metrics['sys_TP'] + sys_metrics['sys_FN'] + epsilon)

print("\n" + "="*65)
print(" 📊 OLS-BASED PERFORMANCE METRICS REPORT")
print("="*65)
print(f"🔹 OLS Thresholds Evaluated : 0.50 to 0.90 (step 0.05)")
print(f"🔹 Confidence Thresh        : {CONF_THRESH * 100}%")
print("="*65)

print("\n--- 1. OVERALL METRICS (Mean over all OLS Thresholds) ---")
print(f" Average Recall (AR)    : {final_AR * 100:>6.2f} %")
print(f" Average Precision (AP) : {final_AP * 100:>6.2f} %")

print(f"\n--- 2. SYSTEM-LEVEL METRICS ({WINDOW_SIZE}-Frame Window @ OLS=0.50) ---")
print(f" System Pd              : {Sys_Recall * 100:>6.2f} %")
print(f" System Precision       : {Sys_Precision * 100:>6.2f} %")
print(f" Confusion Matrix       : TP={sys_metrics['sys_TP']} | FP={sys_metrics['sys_FP']} | FN={sys_metrics['sys_FN']} | TN={sys_metrics['sys_TN']}")
print("="*65)


 🚀 STARTING OLS-BASED METRICS EVALUATION ...
 ✅ Successfully loaded weights from 'adas_radar_weights_best.pth'


Evaluating Frames: 100%|██████████| 1795/1795 [00:21<00:00, 81.63it/s]


 📊 OLS-BASED PERFORMANCE METRICS REPORT
🔹 OLS Thresholds Evaluated : 0.50 to 0.90 (step 0.05)
🔹 Confidence Thresh        : 80.0%

--- 1. OVERALL METRICS (Mean over all OLS Thresholds) ---
 Average Recall (AR)    :  95.63 %
 Average Precision (AP) :  97.82 %

--- 2. SYSTEM-LEVEL METRICS (5-Frame Window @ OLS=0.50) ---
 System Pd              :  99.72 %
 System Precision       : 100.00 %
 Confusion Matrix       : TP=358 | FP=0 | FN=1 | TN=0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# 1. create base folder
!mkdir -p /content/radar_data

# 2. unzip the main dataset (large)
print("Unzipping main dataset (Tensors & Old Labels)...")
!unzip -q "/content/drive/MyDrive/Dataset/Automotive.zip" -d /content/radar_data/Automotive/processed_tensors

# 3. unzip the update (new labels) over the old files with overwrite (-o)
print("Applying new text_labels patch...")
!unzip -o -q "/content/drive/MyDrive/Dataset/Updated_Labels.zip" -d /content/radar_data/Automotive

print(" Dataset is fully updated and ready for training!")

# to verify that the folders exist
!ls /content/radar_data/Automotive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping main dataset (Tensors & Old Labels)...
Applying new text_labels patch...
 Dataset is fully updated and ready for training!
2019_04_09_bms1000  2019_04_30_cm1s000	2019_05_09_bm1s007  2019_05_29_cm1s014
2019_04_09_cms1000  2019_04_30_mlms000	2019_05_09_cm1s003  2019_05_29_mlms006
2019_04_09_css1000  2019_04_30_mlms001	2019_05_09_mlms003  2019_05_29_pbms007
2019_04_09_pms1000  2019_04_30_pbms002	2019_05_09_pbms004  2019_05_29_pcms005
2019_04_09_pms2000  2019_04_30_pbss000	2019_05_09_pcms002  processed_tensors
2019_04_09_pms3000  2019_04_30_pcms001	2019_05_29_bcms000
